# LRU Cache

**Company:** MongoDB (GothamLoop question bank) · **Category:** Coding · **Tags:** Live Screen, Caching, Hash Tables, Linked List · **Difficulty/Frequency:** Common (5/10)

> **Related:** [`12. Linked_Hash_Map`](../12.%20Linked_Hash_Map/12.%20Linked_Hash_Map.ipynb) is the same structure in *insertion* order with no capacity bound. An LRU cache is a linked hash map in **access** order, plus an eviction rule.

## Concepts

**What this problem is really testing:**
- Recognising that **one data structure cannot do this** — and composing two that can
- Why the hash map must store **node references**, not values
- Pointer surgery on a **doubly linked list**, made safe by sentinels

**First-principles primer — what is each piece?**

- **Cache.** A small, fast store in front of a slow one. It is *bounded*, which forces the real question: when it is full and something new arrives, **what do you throw away?**
- **LRU (least recently used).** The eviction policy: discard whatever has gone longest without being touched. The bet is *temporal locality* — something used recently is likely to be used again soon. It is the default policy almost everywhere (CPU caches, page caches, MongoDB's WiredTiger cache) because it is cheap and usually right.
- **Doubly linked list.** Nodes with `prev` **and** `next` pointers. The second pointer is what makes it work: with only `next`, unlinking a node requires walking from the head to find its predecessor — O(n). With `prev`, you can splice a node out in **four assignments**, no searching.
- **Sentinel nodes.** Two permanent dummy nodes that are never real entries, parked at each end. Because every real node then always has a `prev` and a `next`, the "is this the first/last one?" branches disappear entirely.

**The insight — two structures, two questions:**

| Question | Structure | Cost |
|---|---|---|
| "What is the value for key K?" | hash map | O(1) |
| "Which entry is least recently used?" | doubly linked list, ordered by recency | O(1) — it is the tail |

Neither alone is enough. A hash map has no order. A list has no fast lookup. **Together** they cover both, and the join between them is the crucial design decision:

> **The map stores `key -> Node`, not `key -> value`.**

That is the whole trick. If the map stored values, then moving a touched entry to the front would mean *searching the list* for its node — O(n), and the entire design collapses. Because the map hands you the node **directly**, you can splice it out and re-insert it at the head in constant time.

**Why the head/tail sentinels matter.** Without them, `_remove` has to ask "is this the first node? the last? the only one?" — and each of those is a branch you can get wrong under interview pressure. With them, every real node is guaranteed to have neighbours, so removal is unconditionally:

```
node.prev.next = node.next
node.next.prev = node.prev
```

**Simple worked example.** Capacity 2:

| operation | map | list (head → tail) | note |
|---|---|---|---|
| `put(1,"a")` | `{1}` | `1` | |
| `put(2,"b")` | `{1,2}` | `2, 1` | newest at the head |
| `get(1)` → `"a"` | `{1,2}` | `1, 2` | touching 1 moves it to the front |
| `put(3,"c")` | `{1,3}` | `3, 1` | full → evict the **tail**, which is 2 |
| `get(2)` → `-1` | | | 2 is gone — and note `get(1)` is what saved 1 |

## Problem Statement

Implement an `LRUCache(capacity)`:

| Method | Behaviour |
|---|---|
| `get(key)` | Return the value, **and mark the key as most recently used**. Return `-1` if absent |
| `put(key, value)` | Insert or update. If this pushes the size over `capacity`, **evict the least recently used** entry |

Both operations must run in **O(1)** average time.

**Example**

```python
c = LRUCache(2)
c.put(1, 1)
c.put(2, 2)
c.get(1)       # -> 1     (1 is now the most recent)
c.put(3, 3)    # evicts key 2, the least recently used
c.get(2)       # -> -1
c.put(4, 4)    # evicts key 1
c.get(1)       # -> -1
c.get(3)       # -> 3
c.get(4)       # -> 4
```

### Approach 1 — Naive (a list of key/value pairs in recency order)

**Idea:** keep entries in a plain list, most recent at index 0. `get` scans for the key, then moves it to the front. `put` scans, then inserts at the front, popping the last element when full.

Correct and readable — and O(n) on **every single operation**, because `list.remove` and `list.insert(0, ...)` both scan and shift. It is the honest baseline to state before improving on it.

**Time complexity:** **O(n)** per `get` and `put`.

**Space complexity:** O(capacity).

In [ ]:
from typing import Any, Dict, List, Optional, Tuple


class NaiveLRUCache:
    """Baseline: correct LRU semantics, O(n) per operation."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.entries: List[List[Any]] = []          # [key, value], most recent first

    def get(self, key: Any) -> Any:
        for i, pair in enumerate(self.entries):     # O(n) scan
            if pair[0] == key:
                self.entries.insert(0, self.entries.pop(i))   # O(n) shift
                return pair[1]
        return -1

    def put(self, key: Any, value: Any) -> None:
        for i, pair in enumerate(self.entries):     # O(n) scan
            if pair[0] == key:
                pair[1] = value
                self.entries.insert(0, self.entries.pop(i))
                return
        self.entries.insert(0, [key, value])        # O(n) shift
        if len(self.entries) > self.capacity:
            self.entries.pop()                      # the tail is least recently used

### Approach 2 — Optimal (hash map of nodes + doubly linked list)

**Idea:** the list carries the *order*, the map carries the *lookup*, and they share the very same node objects.

- `cache: key -> Node` — O(1) to find any entry
- `head <-> ... <-> tail` — most recent behind `head`, least recent before `tail`

Every operation is then a fixed number of pointer assignments:

- **`get(key)`** — map lookup, unlink the node, re-insert at the head, return its value.
- **`put(key, value)`, key present** — update the value, move the node to the head. **The size does not change** — this is the case people forget, and forgetting it inserts a *second* node for the same key, corrupting the list.
- **`put(key, value)`, key absent** — create a node, insert at head, add to the map; if the size now exceeds capacity, pop the **tail** node and delete its key from the map.

That last detail is why every node stores its **own key** even though the map is already keyed by it: when you evict the tail you hold a *node*, and you need its key to remove the matching map entry. Without `node.key` you would have to search the map by value — O(n), and the whole design falls over.

**Time complexity:** **O(1)** average for both operations — one hash lookup plus a constant number of pointer writes.

**Space complexity:** **O(capacity)** — the map and the list hold *the same* nodes, so it is O(capacity), not O(2 × capacity).

In [ ]:
class Node:
    __slots__ = ("key", "value", "prev", "next")

    def __init__(self, key: Any = None, value: Any = None) -> None:
        self.key = key            # the node stores its OWN key, so eviction can clean the map
        self.value = value
        self.prev: Optional["Node"] = None
        self.next: Optional["Node"] = None


class LRUCache:
    """Hash map of nodes + doubly linked list. O(1) get and put."""

    def __init__(self, capacity: int) -> None:
        if capacity < 0:
            raise ValueError("capacity must be non-negative")
        self.capacity = capacity
        self.cache: Dict[Any, Node] = {}          # key -> the SAME node that is in the list
        self.head = Node()                        # sentinel: most-recent side
        self.tail = Node()                        # sentinel: least-recent side
        self.head.next = self.tail
        self.tail.prev = self.head

    # ---- list primitives: no branches, thanks to the sentinels ----------
    def _remove(self, node: Node) -> None:
        node.prev.next = node.next                # both neighbours always exist
        node.next.prev = node.prev

    def _add_to_head(self, node: Node) -> None:
        node.prev = self.head
        node.next = self.head.next
        self.head.next.prev = node
        self.head.next = node

    def _move_to_head(self, node: Node) -> None:
        self._remove(node)
        self._add_to_head(node)

    def _pop_tail(self) -> Node:
        node = self.tail.prev                     # the real node just before the tail sentinel
        self._remove(node)
        return node

    # ---- public API ------------------------------------------------------
    def get(self, key: Any) -> Any:
        node = self.cache.get(key)
        if node is None:
            return -1
        self._move_to_head(node)                  # reading COUNTS as a use
        return node.value

    def put(self, key: Any, value: Any) -> None:
        if self.capacity == 0:
            return                                # a zero-capacity cache stores nothing
        node = self.cache.get(key)
        if node is not None:
            node.value = value                    # UPDATE: size is unchanged...
            self._move_to_head(node)              # ...but it is now the most recent
            return
        node = Node(key, value)
        self.cache[key] = node
        self._add_to_head(node)
        if len(self.cache) > self.capacity:
            evicted = self._pop_tail()
            del self.cache[evicted.key]           # THIS is why the node remembers its key

    # ---- introspection, for the tests below ------------------------------
    def keys_mru_first(self) -> List[Any]:
        out, node = [], self.head.next
        while node is not self.tail:
            out.append(node.key)
            node = node.next
        return out

    def __len__(self) -> int:
        return len(self.cache)

### Approach 3 — `OrderedDict` (the same structure, from the standard library)

**Idea:** `collections.OrderedDict` *is* a hash map threaded with a doubly linked list — exactly the design above, implemented in C. `move_to_end` relinks a node; `popitem(last=False)` pops the oldest.

Worth knowing for two opposite reasons:

- **In production, use this.** Fewer lines, no pointer bugs, and the relinking happens in C.
- **In an interview, do not lead with it.** The question is asking whether you can *build* the structure. Write Approach 2, then mention this as what you would actually ship — that ordering shows both capability and judgement.

(`functools.lru_cache` is this same machinery again, wrapped as a decorator.)

**Time complexity:** O(1) average per operation.

**Space complexity:** O(capacity).

In [ ]:
from collections import OrderedDict


class OrderedDictLRUCache:
    """The same algorithm, using the standard library's built-in linked hash map."""

    def __init__(self, capacity: int) -> None:
        self.capacity = capacity
        self.od: "OrderedDict[Any, Any]" = OrderedDict()

    def get(self, key: Any) -> Any:
        if key not in self.od:
            return -1
        self.od.move_to_end(key)                  # relink this node to the most-recent end
        return self.od[key]

    def put(self, key: Any, value: Any) -> None:
        if self.capacity == 0:
            return
        if key in self.od:
            self.od.move_to_end(key)
        self.od[key] = value
        if len(self.od) > self.capacity:
            self.od.popitem(last=False)           # drop the least recently used

### Follow-up — LRU with a TTL (time-to-live)

**Idea:** entries should also expire on their own after `ttl` seconds, whether or not the cache is full. Two eviction reasons now coexist: **capacity** (LRU) and **age** (TTL).

The design question is *when* to notice an expiry. Scanning the whole cache on every access would be O(n) and destroy the point. The standard answer is **lazy expiry**: store an `expires_at` on each node and check it *only when that key is touched*. An expired entry is deleted on the spot and reported as a miss.

Lazy expiry costs O(1) and is what Redis actually does. Its one weakness is that an expired key nobody ever touches keeps occupying memory — so real systems pair it with a periodic background sweep, or simply let LRU eviction reclaim it eventually.

Note that `get` must check expiry **before** promoting the node, or you would refresh the recency of something already dead.

**Time complexity:** O(1) per operation (amortised — an expired entry is cleaned up by the access that finds it).

**Space complexity:** O(capacity), plus one timestamp per entry.

In [ ]:
import time


class TTLCache(LRUCache):
    """LRU eviction by capacity, plus lazy expiry by age."""

    def __init__(self, capacity: int, ttl: float) -> None:
        super().__init__(capacity)
        self.ttl = ttl
        self.expires: Dict[Any, float] = {}       # key -> monotonic deadline

    def _expired(self, key: Any, now: float) -> bool:
        return key in self.expires and now >= self.expires[key]

    def get(self, key: Any, now: Optional[float] = None) -> Any:
        now = time.monotonic() if now is None else now
        node = self.cache.get(key)
        if node is None:
            return -1
        if self._expired(key, now):               # check BEFORE promoting - never refresh the dead
            self._remove(node)
            del self.cache[key]
            del self.expires[key]
            return -1
        self._move_to_head(node)
        return node.value

    def put(self, key: Any, value: Any, now: Optional[float] = None) -> None:
        now = time.monotonic() if now is None else now
        super().put(key, value)
        if key in self.cache:                     # capacity 0 stores nothing
            self.expires[key] = now + self.ttl

    def _pop_tail(self) -> Node:
        node = super()._pop_tail()
        self.expires.pop(node.key, None)          # keep the two maps in step on eviction
        return node

## Verification

Run the canonical LRU trace, then the cases that actually break implementations: `get` counting as a use, `put` on an existing key **not** growing the cache, capacity 1 and 0, and — most importantly — a direct check that the map and the list never disagree.

In [ ]:
import random

IMPLS = [LRUCache, NaiveLRUCache, OrderedDictLRUCache]

# --- The canonical trace, on all three implementations ---
for cls in IMPLS:
    c = cls(2)
    c.put(1, 1)
    c.put(2, 2)
    assert c.get(1) == 1, cls.__name__            # 1 becomes most recent
    c.put(3, 3)                                   # -> evicts 2, NOT 1
    assert c.get(2) == -1, cls.__name__
    c.put(4, 4)                                   # -> evicts 1
    assert c.get(1) == -1, cls.__name__
    assert c.get(3) == 3, cls.__name__
    assert c.get(4) == 4, cls.__name__

# --- get() must COUNT as a use; without that, LRU is just FIFO ---
for cls in IMPLS:
    c = cls(2)
    c.put(1, 1)
    c.put(2, 2)
    c.get(1)                                      # touching 1 protects it
    c.put(3, 3)
    assert c.get(1) == 1, f"{cls.__name__}: get() must refresh recency"
    assert c.get(2) == -1, f"{cls.__name__}: 2 was the least recently used"

# --- put() on an EXISTING key updates in place: it must not grow the cache ---
for cls in IMPLS:
    c = cls(2)
    c.put(1, "a")
    c.put(2, "b")
    c.put(1, "A")                                 # update, not insert
    assert c.get(1) == "A", cls.__name__
    assert c.get(2) == "b", f"{cls.__name__}: updating 1 must not have evicted 2"
c = LRUCache(2)
c.put(1, "a"); c.put(2, "b"); c.put(1, "A")
assert len(c) == 2 and c.keys_mru_first() == [1, 2], "an update must not duplicate the node"

# --- Degenerate capacities ---
for cls in IMPLS:
    c = cls(1)
    c.put(1, 1)
    assert c.get(1) == 1, cls.__name__
    c.put(2, 2)                                   # capacity 1: every insert evicts
    assert c.get(1) == -1 and c.get(2) == 2, cls.__name__

    z = cls(0)
    z.put(1, 1)
    assert z.get(1) == -1, f"{cls.__name__}: a zero-capacity cache stores nothing"

# --- The list order is exactly what LRU says it should be ---
c = LRUCache(3)
for k in (1, 2, 3):
    c.put(k, k)
assert c.keys_mru_first() == [3, 2, 1]
c.get(1)
assert c.keys_mru_first() == [1, 3, 2]
c.put(2, 22)
assert c.keys_mru_first() == [2, 1, 3]
c.put(4, 4)                                       # full -> evict the tail, which is 3
assert c.keys_mru_first() == [4, 2, 1]
assert c.get(3) == -1

# --- The map and the list must never disagree ---
def check_consistency(cache: LRUCache) -> None:
    forward, node = [], cache.head.next
    while node is not cache.tail:
        forward.append(node)
        node = node.next
    backward, node = [], cache.tail.prev          # walking back must mirror walking forward
    while node is not cache.head:
        backward.append(node)
        node = node.prev
    assert forward == backward[::-1], "prev/next pointers disagree - the list is corrupt"
    assert len(forward) == len(cache.cache), "the list and the map hold different counts"
    for n in forward:
        assert cache.cache[n.key] is n, "the map must point at the very node in the list"
    assert len(cache.cache) <= cache.capacity, "capacity exceeded"


# --- Randomised: the fast cache must behave identically to the naive one ---
random.seed(13)
for capacity in (1, 2, 3, 8):
    fast, naive, od = LRUCache(capacity), NaiveLRUCache(capacity), OrderedDictLRUCache(capacity)
    for step in range(3000):
        k = random.randrange(12)
        if random.random() < 0.5:
            v = random.randrange(100)
            fast.put(k, v); naive.put(k, v); od.put(k, v)
        else:
            a, b, d = fast.get(k), naive.get(k), od.get(k)
            assert a == b == d, (capacity, step, k, a, b, d)
        if step % 100 == 0:
            check_consistency(fast)
    check_consistency(fast)

# --- TTL follow-up: expiry is lazy, and independent of capacity ---
t = TTLCache(capacity=3, ttl=10.0)
t.put("a", 1, now=0.0)
t.put("b", 2, now=0.0)
assert t.get("a", now=5.0) == 1                   # still fresh
assert t.get("a", now=11.0) == -1, "an entry past its TTL must be a miss"
assert "a" not in t.cache, "the expired entry is deleted on the access that finds it"
assert t.get("b", now=11.0) == -1
assert len(t) == 0

t = TTLCache(capacity=2, ttl=10.0)
t.put("x", 1, now=0.0)
t.put("x", 2, now=8.0)                            # re-putting must refresh the deadline
assert t.get("x", now=15.0) == 2, "put() resets the TTL"
t.put("y", 1, now=15.0)
t.put("z", 1, now=15.0)                           # capacity 2 -> LRU still evicts, TTL or not
assert t.get("x", now=16.0) == -1
assert "x" not in t.expires, "eviction must clean the expiry map too"

print("All assertions passed.")

## Discussion — remaining follow-up directions

- **Thread safety.** A single `Lock` around `get` and `put` is correct and, crucially, *sufficient* — both operations are already O(1), so the critical section is tiny and contention is low. The catch is that **`get` is a writer**: it relinks the list, so you cannot use a readers–writer lock and let reads run in parallel. That is a genuinely surprising property of LRU, and it is why high-throughput caches **shard** — 16 independent caches keyed by `hash(key) % 16`, each with its own lock, so unrelated keys never contend. The cost is that the eviction policy becomes per-shard rather than global.
- **A `peek` that does not refresh recency.** Easy to add (skip `_move_to_head`) but think about what it *means*: LRU's whole premise is that a read predicts a future read. A `peek` is a read you are declaring non-predictive — legitimate for debugging, monitoring, or bulk scans that would otherwise flush every genuinely hot key out of the cache. That last case is real: a full table scan through an LRU cache evicts everything useful, which is exactly why databases give scans their own cache policy.
- **Persistence across restarts.** Two options. A **snapshot** (serialise `keys_mru_first()` plus the values, periodically) is simple and loses at most one interval's worth. A **write-ahead log** of `put`/evict operations, replayed on startup, loses nothing but costs a disk write per operation — see [`3. Persistent_Append_Only_Log`](../3.%20Persistent_Append_Only_Log/3.%20Persistent_Append_Only_Log.ipynb) for exactly that machinery. Worth asking first *whether a cache should be durable at all*: it is a performance optimisation over a source of truth, so a cold start is usually acceptable and rebuilding is often cheaper than persisting.
- **Beyond LRU.** LRU has a known weakness — one sequential scan touches everything once and evicts your entire working set. **LFU** (least *frequently* used) resists that but adapts slowly to genuine change. **LRU-K** tracks the last K accesses rather than just the most recent, and **ARC** adaptively balances recency against frequency. Naming *why* LRU fails on scans is what shows you understand the policy rather than just its implementation.
- **Why the node stores its own key.** Worth saying explicitly, because it looks redundant next to a map already keyed by it. On eviction you hold a *node* and must delete the corresponding *map entry* — without `node.key` that is a search by value, O(n), and the O(1) claim evaporates. This is the standard trick for any structure where two containers share objects: **each object carries whatever its peers need to find it.**

## Empirical complexity check

Compare the **list-based** cache (Approach 1, O(n) per operation) with the **map + linked list** (Approach 2, O(1)) — running a fixed number of operations against a cache whose **capacity** doubles.

Capacity is the variable that matters: the naive version's scan length is bounded by how many entries it holds, while the optimal version does the same handful of pointer writes no matter how large the cache is.

| Growth when capacity doubles | What it means |
|---|---|
| ~2x | linear — every operation scans the whole cache |
| ~1x | constant — a hash lookup plus a fixed number of pointer assignments |

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark

import random

OPS = 20000


def make_workload(capacity):
    rng = random.Random(17)
    # Keys drawn from twice the capacity => a realistic mix of hits and evictions.
    ops = [(rng.random() < 0.5, rng.randrange(capacity * 2)) for _ in range(OPS)]
    return (capacity, ops)


def _drive(cache, ops):
    for is_put, k in ops:
        if is_put:
            cache.put(k, k)
        else:
            cache.get(k)


def run_naive(capacity, ops):
    _drive(NaiveLRUCache(capacity), ops)          # O(n) scan + shift per operation


def run_optimal(capacity, ops):
    _drive(LRUCache(capacity), ops)               # O(1): hash lookup + pointer writes


def run_ordereddict(capacity, ops):
    _drive(OrderedDictLRUCache(capacity), ops)    # same algorithm, implemented in C


benchmark(
    {"Approach 1 - list scan O(n)": run_naive,
     "Approach 2 - map + linked list O(1)": run_optimal,
     "Approach 3 - OrderedDict O(1), in C": run_ordereddict},
    make_workload,
    sizes=[250, 500, 1000, 2000],
    repeats=2,
)

## Patterns learned

- **When no single structure answers both questions, compose two — and let them share the objects.** Hash map for *identity*, linked list for *order*. This pairing is the whole answer, and it recurs constantly: an LFU cache, a linked hash map, a timer wheel, a scheduler queue.
- **Store references, not copies.** The map holds the very node the list holds. That shared ownership is what makes the cross-structure operation O(1) instead of a search — and it is why the space is O(capacity), not O(2 × capacity).
- **Give each shared object whatever its peers need to find it.** `node.key` looks redundant until eviction, where it is the difference between O(1) and O(n).
- **Sentinels turn branches into straight-line code.** Two dummy nodes remove every "is this the first/last/only element?" case. Fewer branches, fewer bugs, and code you can write correctly on a whiteboard.
- **A doubly linked list buys you O(1) removal from the middle.** That backward pointer is the entire reason to prefer it over a singly linked list here.
- **State the invariant, then check every method against it.** *"The list holds exactly the map's keys, ordered most-recent first, and its length never exceeds capacity."* The `check_consistency` helper above turns that sentence directly into a test.
- **Know the standard-library shortcut, and know when to reach for it.** `OrderedDict` / `functools.lru_cache` are this structure in C. Build it by hand when asked to build it; ship the library version.